# Forecast Uncertainty

In the previous lecture, we compared forecasting procedures, selected one, and issued a forecast. A tourism planner could use that outlook to prepare for next year's travel. The next question is how much room to allow around the predicted values.

A point forecast gives us a useful center. A prediction interval adds a range of possible outcomes. Today we will calculate that range, explain why it changes with the forecast horizon, and construct it a second way using the bootstrap.

We will use a familiar example: Google's daily closing prices from 11b. The latest-value forecast makes the calculations simple enough to follow by hand. The same ideas about forecast errors and possible futures apply to the more elaborate methods we have studied.

## References

This lecture follows the normal-theory and residual-bootstrap topics in [FPP Pythonic §5.5, Distributional Forecasts and Prediction Intervals](https://otexts.com/fpppy/05-toolbox.html#distributional-forecasts-and-prediction-intervals), by Rob J. Hyndman, George Athanasopoulos, Azul Garza, Cristian Challu, Max Mergenthaler, and Kin G. Olivares. The explanations and worked calculations below connect those topics to our earlier inference, random-walk, and forecasting lectures.

[StatsForecast's Naive model](https://nixtlaverse.nixtla.io/statsforecast/src/core/models.html#naive) supplies the library forecasts. [Data provenance](../11b-autocorrelation-and-stationarity/data/README.md) documents the frozen 2015 Google extract. The course environment includes StatsForecast; in Colab, run `%pip install statsforecast==2.0.1` first.

## Prediction Intervals

### A Range of Possible Outcomes

Suppose our next-quarter forecast is 12.4 million holiday trips. A future count of 12.0 million and one of 9.0 million would have very different planning consequences. A forecast distribution assigns probabilities to possible future values, using the observed history and a model of how new observations arise.

An 80% prediction interval takes the middle 80% of that distribution, leaving 10% in each tail. A 95% interval leaves 2.5% in each tail and covers a wider range. These probabilities describe the forecast distribution supplied by the model. When later observations arrive, we can check how often the intervals contain them.

### Confidence Intervals

Recall the confidence intervals we used earlier in the course. We estimated a population mean using a sample mean, then allowed for how much that estimate could vary across repeated samples.

A familiar confidence-interval formula is

$$
\bar x\pm t^*\frac{s}{\sqrt n}
$$

The three parts have different jobs:

- $\bar x$ is our estimate of the population mean
- $s/\sqrt n$ is the estimated **standard error of the sample mean**, calculated from the sample standard deviation $s$ and sample size $n$
- $t^*$ is the multiplier for the chosen confidence level and degrees of freedom

A **standard error is a standard deviation**: it describes the sampling distribution of an estimator. Imagine repeatedly collecting an independent sample of the same size and calculating its mean. The standard deviation of those means is the standard error; $s/\sqrt n$ estimates it.

For example, if individual measurements have a standard deviation of about 10, a mean based on 100 independent measurements has an estimated standard error of

$$
\frac{s}{\sqrt n}=\frac{10}{\sqrt{100}}=1
$$

Averaging makes the sample mean less variable than an individual measurement. Increasing the sample size improves our estimate of the population mean, while individual measurements can still vary substantially around that mean.

We can write the interval's general structure as

$$
\text{estimate}\;\pm\;\text{multiplier}\times\text{standard deviation of the estimation error}
$$

### Forecast Errors

For forecasting, we use the same general construction, but we need the variation in a different error:

$$
\text{forecast error}=\text{future observation}-\text{point forecast}
$$

Suppose we have observed the series through time $T$ and want to predict $h$ steps ahead. Write the future observation as $y_{T+h}$ and its forecast as $\hat y_{T+h|T}$. The part after the vertical bar records when the forecast was issued.

Even if our model and its parameters were known exactly, new random shocks would make the future observation uncertain. The relevant spread is how much that observation could differ from its forecast.

Write the estimated standard deviation of this forecast error as $\hat\sigma_h$. The subscript reminds us that its size can depend on how far ahead we are forecasting. A symmetric prediction interval can be written as

$$
\hat y_{T+h|T}\pm c\,\hat\sigma_h
$$

The pieces now have these jobs:

- $\hat y_{T+h|T}$ supplies the point forecast
- $\hat\sigma_h$ measures the spread of forecast errors at horizon $h$
- $c$ is a multiplier determined by the chosen coverage and assumptions about the forecast-error distribution

For normally distributed forecast errors, $c=1.96$ gives the central 95% interval. For a central 80% interval, $c\approx1.28$.

Our next task is to calculate that forecast-error standard deviation: first for one step ahead, then for longer horizons.

### The Google Price Series

Load the 252 trading observations from 2015 and calculate each change from the previous trading observation. We will count steps in trading observations, preserving their order. Weekends and market holidays contribute no extra rows.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm
from statsforecast import StatsForecast
from statsforecast.models import Naive

candidates = [
    Path("../11b-autocorrelation-and-stationarity/data/google-2015.csv"),
    Path("content/lectures/11b-autocorrelation-and-stationarity/data/google-2015.csv"),
    Path("lectures/11b-autocorrelation-and-stationarity/data/google-2015.csv"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is not None:
    prices = pd.read_csv(data_path, parse_dates=["ds"])
else:
    data_url = (
        "https://raw.githubusercontent.com/olearydj/INSY7130/main/"
        "lectures/11b-autocorrelation-and-stationarity/data/google-2015.csv"
    )
    with urlopen(data_url, timeout=30) as response:
        prices = pd.read_csv(response, parse_dates=["ds"])

data = pd.DataFrame({
    "unique_id": "Google",
    "ds": np.arange(1, len(prices) + 1),
    "y": prices["Close"],
})
assert len(data) == 252 and prices["ds"].is_monotonic_increasing
assert prices["ds"].is_unique and data["y"].notna().all()
pd.set_option("display.precision", 2)
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})

display(data.head())

The table uses the same three-column layout as Tuesday's StatsForecast example. Here `ds` is the trading-observation number, and `y` is the closing price in dollars.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(prices["ds"], data["y"], color="C0")
axes[0].set(title="Google Closing Prices, 2015", xlabel="Date", ylabel="Price (USD)")
axes[1].plot(prices["ds"].iloc[1:], data["y"].diff().dropna(), color="C1")
axes[1].axhline(0, color="gray", linewidth=1)
axes[1].set(title="Changes Between Trading Observations", xlabel="Date",
            ylabel="Price change (USD)")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()
plt.close(fig)

The level moves substantially through the year. Changes fluctuate around zero, with a few large moves. We will use these changes to describe the shocks that a latest-value forecast leaves unpredictable.

## Normal-Theory Intervals

### Estimate the One-Step Error Scale

The naive method predicts the next price using the latest observed price:

$$
\hat y_{t|t-1}=y_{t-1}
$$

Its one-step error is therefore the observed change:

$$
e_t=y_t-\hat y_{t|t-1}=y_t-y_{t-1}
$$

Here, each new observation equals the previous value plus an unpredictable contribution called a **shock**, or **innovation**. Because the forecast is the previous value, the one-step prediction error gives us the realized shock. Here, that is simply the change between successive prices.

For example, if yesterday's price was `$750` and today's is `$758`, yesterday's forecast missed by `$8`. A decrease of `$8` would produce an error of `−$8`.

Under a model with zero-mean shocks, estimate their variance by averaging the squared one-step errors. Taking the square root returns to dollars:

$$
\hat\sigma=\sqrt{\frac{1}{T-1}\sum_{t=2}^{T}e_t^2}
$$

With $T$ observed prices, we have $T-1$ changes. The first price has no preceding observation for calculating an error. This zero-mean root-mean-square calculation is also the scale used by StatsForecast's `Naive` intervals.

In [ ]:
errors = data["y"].diff().dropna().to_numpy()
sigma = np.sqrt(np.mean(errors**2))
last_price = data["y"].iloc[-1]
print(f"Last observed price: ${last_price:.2f}")
print(f"One-step error scale: ${sigma:.2f} from {len(errors)} changes")

`.diff()` subtracts the preceding price from each price; `.dropna()` removes the first, undefined difference. Squaring, averaging, and taking the square root gives an estimated one-step error scale of `$11.19`. The latest closing price is `$758.88`.

### Calculate a One-Step Interval

Assume the next shock follows a normal distribution with mean zero and standard deviation estimated by `$11.19`. About 95% of a normal distribution lies within 1.96 standard deviations of its mean. Add and subtract that allowance from the point forecast:

$$
\hat y_{T+1|T}\pm1.96\hat\sigma
\;\approx\;758.88\pm1.96(11.19)
\;\approx\;[736.95,\;780.81]
$$

The interval gives a plausible range for the next closing price under this model. For an 80% interval, the multiplier is about 1.28, so the range is narrower.

Calculate both intervals using the normal quantiles at full precision.

In [ ]:
z80 = norm.ppf(0.90)
z95 = norm.ppf(0.975)
one_step = pd.DataFrame({
    "Level": ["80%", "95%"],
    "Multiplier": [z80, z95],
    "Lower (USD)": last_price - np.array([z80, z95]) * sigma,
    "Forecast (USD)": last_price,
    "Upper (USD)": last_price + np.array([z80, z95]) * sigma,
})
display(one_step.round(2))

`norm.ppf()` returns a standard-normal quantile. For the central 95%, the upper endpoint is at cumulative probability 0.975; the remaining 0.025 is the upper tail. The 80% interval is about `$744.54`–`$773.22`.

### Random Walks and Longer Horizons

Recall the random walk from 11b. Each observation equals the preceding observation plus a new shock:

$$
y_{T+1}=y_T+\varepsilon_{T+1}
$$

Here, $\varepsilon_{T+1}$ denotes the next shock. Its value is unknown when we issue the forecast.

If the shocks have mean zero, our best mean prediction for the next observation is the current value. The expected change is zero. Repeating that reasoning gives the same point forecast at every horizon.

The possible outcomes spread out, however. Two steps ahead, substitute the first update into the second:

$$
y_{T+2}=y_{T+1}+\varepsilon_{T+2}
=y_T+\varepsilon_{T+1}+\varepsilon_{T+2}
$$

The two-step forecast error contains two unknown shocks. For independent shocks, each with variance $\sigma^2$, their variances add:

$$
\operatorname{Var}(y_{T+2}-y_T)=2\sigma^2
\qquad\Longrightarrow\qquad
\sigma_2=\sigma\sqrt2
$$

At horizon $h$, there are $h$ future shocks. Under the independent normal-shock model:

$$
\hat y_{T+h|T}=y_T,
\qquad \hat\sigma_h=\hat\sigma\sqrt h,
\qquad \text{95\% interval: }y_T\pm1.96\hat\sigma\sqrt h
$$

At ten steps, the estimated error scale is $\hat\sigma_{10}=11.19\sqrt{10}\approx35.38$ dollars. The 95% interval becomes approximately `$689.53`–`$828.23`. A flat forecast can have growing uncertainty because more shocks accumulate before the target arrives.

The next calculation keeps the point forecast fixed and changes only the horizon.

In [ ]:
horizon = 20
steps = np.arange(1, horizon + 1)
sigma_h = sigma * np.sqrt(steps)
manual = pd.DataFrame({
    "Horizon": steps,
    "Forecast (USD)": last_price,
    "Error SD (USD)": sigma_h,
    "95% lower (USD)": last_price - z95 * sigma_h,
    "95% upper (USD)": last_price + z95 * sigma_h,
})
display(manual.loc[manual["Horizon"].isin([1, 2, 5, 10, 20])].round(2))

Each row refers to a forecast issued from the same final observation. Ten steps ahead has about 3.16 times the one-step error standard deviation. This square-root rule follows from the random-walk model; other forecasting models propagate shocks differently.

How many steps ahead would make this interval twice as wide as the one-step interval?

##### Answer

Four steps. The width is proportional to $\sqrt h$, and $\sqrt4=2$. Doubling the horizon to two would multiply the width by about 1.41.

### Generate the Intervals With StatsForecast

Use the same `StatsForecast` interface as Tuesday, adding `level=[80, 95]`. The library will return the point forecast and both sets of bounds.

In [ ]:
forecaster = StatsForecast(models=[Naive()], freq=1)
forecast = forecaster.forecast(df=data, h=horizon, level=[80, 95])
display(forecast.head().round(2))

`h=20` requests the next twenty trading observations. `freq=1` advances the observation number by one. `Naive` contains the point forecast; `Naive-lo-95` and `Naive-hi-95` contain the lower and upper 95% endpoints. The corresponding `80` columns contain the 80% interval. The first row reproduces our one-step calculation.

Plot these bounds around the forecast. Start with the final sixty historical observations so the future range is easy to inspect.

In [ ]:
fig, ax = plt.subplots()
recent = data.tail(60)
ax.plot(recent["ds"], recent["y"], color="black", label="Observed price")
ax.fill_between(forecast["ds"], forecast["Naive-lo-95"], forecast["Naive-hi-95"],
                color="C0", alpha=0.16, label="95% interval")
ax.fill_between(forecast["ds"], forecast["Naive-lo-80"], forecast["Naive-hi-80"],
                color="C0", alpha=0.35, label="80% interval")
ax.plot(forecast["ds"], forecast["Naive"], color="C0", label="Point forecast")
ax.axvline(data["ds"].iloc[-1], color="gray", linestyle=":")
ax.set(title="One Forecast Origin, Increasing Uncertainty",
       xlabel="Trading observation", ylabel="Price (USD)")
ax.legend(loc="upper left", fontsize=10)
plt.tight_layout()
plt.show()
plt.close(fig)

Both bands widen while the point forecast stays at `$758.88`. The lighter band includes the darker one because 95% covers more of the forecast distribution. Each interval applies to its own horizon; a 95% interval at every horizon does not give a 95% probability that an entire future path stays inside all the bands.

## Bootstrap Prediction Intervals

### Reuse the Observed Shocks

The normal calculation describes future shocks through a bell-shaped distribution and an estimated scale. We can also use the observed changes themselves to generate possible futures.

Recall our earlier bootstrap: resample observed values with replacement, repeat a procedure, and inspect the resulting distribution. Our historical one-step errors provide observed examples of shocks. We resample those errors and use them to build future paths, with each selected error supplying a possible next shock.

For the zero-drift random walk, first subtract the average historical error from each error. This makes the resampling pool average zero, matching the assumption behind our flat point forecast. Then a repeatedly sampled path has no systematic upward or downward movement on average.

In [ ]:
shock_pool = errors - errors.mean()
print(f"Average historical change: ${errors.mean():.2f}")
print(f"Average centered shock: ${abs(shock_pool.mean()):.2f}")

The average historical change is about `$0.94`. Centering removes that average from the resampling pool while retaining the observed variation and asymmetry. Sampling independent shocks from one pool assumes that these historical changes are representative of future shocks. Serial dependence or changing variability would call for a different resampling design.

### Build Possible Future Paths

Start each path at the last observed price. To generate the next shock, randomly select one value from our centered historical errors, with replacement. This samples from our empirical estimate of the shock distribution.

Add the sampled shock to the current price to obtain the next simulated price. Draw again from the same error pool and repeat, building the path one step at a time.

For example, sampled shocks of `+$8`, `−$3`, and `+$5` would give this path:

| Step | Calculation | Simulated Price |
| --- | --- | --- |
| Start | Last observed price | `$758.88` |
| 1 | 758.88 + 8 | `$766.88` |
| 2 | 766.88 − 3 | `$763.88` |
| 3 | 763.88 + 5 | `$768.88` |

Repeat that process many times. The following code draws 20,000 sequences of twenty shocks and accumulates each sequence into a future price path.

In [ ]:
rng = np.random.default_rng(7130)
n_paths = 20_000
sampled_shocks = rng.choice(shock_pool, size=(n_paths, horizon), replace=True)
paths = last_price + sampled_shocks.cumsum(axis=1)
display(pd.DataFrame(paths[:3, :5],
                     index=["Path 1", "Path 2", "Path 3"],
                     columns=[f"Step {h}" for h in range(1, 6)]).round(2))

Each row is one simulated future; each column is a horizon. `rng.choice()` samples with replacement, so an observed shock may appear several times. `.cumsum(axis=1)` adds shocks along each row: the second price includes the first two shocks, the third includes the first three, and so on. The fixed seed makes the simulation reproducible.

The first six paths show how successive shocks can reinforce or offset one another.

In [ ]:
fig, ax = plt.subplots()
for i in range(6):
    ax.plot(np.r_[0, steps], np.r_[last_price, paths[i]], linewidth=1.4,
            label=f"Path {i + 1}")
ax.axhline(last_price, color="black", linestyle="--", label="Point forecast")
ax.set(title="Six Possible Futures From the Same Starting Price",
       xlabel="Trading observations ahead", ylabel="Simulated price (USD)",
       xticks=[0, 5, 10, 15, 20])
ax.legend(loc="upper left", fontsize=9, ncol=2)
plt.tight_layout()
plt.show()
plt.close(fig)

Every path starts at `$758.88` and then moves as shocks accumulate. Across many paths, prices at a later horizon have more opportunity to spread out. The point forecast remains the expected value under our zero-mean model.

### Take Percentiles at Each Horizon

To obtain a one-step interval, collect the first simulated price from every path. To obtain a twenty-step interval, collect the twentieth price from every path. We take percentiles separately within each of those horizon-specific collections.

The 2.5th and 97.5th percentiles bound the middle 95%. The 10th and 90th percentiles bound the middle 80%.

In [ ]:
quantiles = np.quantile(paths, [0.025, 0.10, 0.90, 0.975], axis=0)
bootstrap = pd.DataFrame({
    "Horizon": steps,
    "80% lower": quantiles[1],
    "80% upper": quantiles[2],
    "95% lower": quantiles[0],
    "95% upper": quantiles[3],
})
display(bootstrap.loc[bootstrap["Horizon"].isin([1, 5, 10, 20])].round(2))

`axis=0` takes each percentile down the rows, pooling the 20,000 paths at a fixed horizon. The bootstrap uses the shape of the observed shock distribution, so its bounds can lie at different distances above and below the point forecast.

Compare the simulated one-step and twenty-step distributions, then compare the bootstrap bounds with the normal-theory bounds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(paths.min(), paths.max(), 80)
axes[0].hist(paths[:, -1], bins=bins, density=True, alpha=0.45,
             label="Twenty steps ahead", color="C1")
axes[0].hist(paths[:, 0], bins=bins, density=True, histtype="step", linewidth=2,
             label="One step ahead", color="C0")
axes[0].axvline(last_price, color="black", linestyle="--")
axes[0].set(title="Simulated Prices at Two Horizons", xlabel="Price (USD)",
            ylabel="Density")
axes[0].legend(fontsize=9)
axes[1].fill_between(steps, bootstrap["95% lower"], bootstrap["95% upper"],
                     color="C1", alpha=0.16, label="Bootstrap 95%")
axes[1].fill_between(steps, bootstrap["80% lower"], bootstrap["80% upper"],
                     color="C1", alpha=0.35, label="Bootstrap 80%")
axes[1].plot(steps, manual["95% lower (USD)"], color="C0", linestyle="--",
             label="Normal-theory 95% bounds")
axes[1].plot(steps, manual["95% upper (USD)"], color="C0", linestyle="--")
axes[1].axhline(last_price, color="black", linewidth=1)
axes[1].set(title="Two Constructions of Forecast Intervals",
            xlabel="Trading observations ahead", ylabel="Price (USD)",
            xticks=[1, 5, 10, 15, 20])
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()
plt.close(fig)

The twenty-step distribution is much wider than the one-step distribution. The bootstrap's upper 95% bound extends farther above the point forecast than its lower bound extends below it. The historical changes include a large positive jump, which can recur in simulated futures. Normal-theory intervals spread uncertainty symmetrically; the bootstrap carries this asymmetry into the forecast distribution.

The two methods share the same zero-drift forecast and historical record. They describe the shock distribution differently. Centering the resampled shocks also makes their variance slightly smaller than the uncentered mean-square scale used above.

If we doubled the number of simulated paths, would you expect the intervals to become half as wide?

##### Answer

More paths make the simulated percentiles more stable. The spread of future prices is set by the shock distribution and the forecast horizon, so it remains roughly the same. This connects to our earlier bootstrap work: increasing the number of resamples improves the numerical approximation to a distribution.

### Bootstrap Intervals With Statsmodels

We can use the same `ExponentialSmoothing` interface introduced in our smoothing lecture. Statsmodels resamples the fitted errors and propagates them through the model to generate possible future paths. We then take percentiles at each horizon, as in our manual example.

Fit simple exponential smoothing to the Google prices, simulate 20,000 future paths, and calculate the middle 95% of the simulated prices at each step.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

smoothing_fit = ExponentialSmoothing(
    data["y"], trend=None, seasonal=None, initialization_method="estimated",
).fit()

# Reproduce the bootstrap draws.
np.random.seed(7130)
smoothing_paths = smoothing_fit.simulate(
    nsimulations=horizon,
    repetitions=20_000,
    anchor="end",
    error="add",
    random_errors="bootstrap",
)
smoothing_bounds = np.quantile(smoothing_paths, [0.025, 0.975], axis=1)
smoothing_intervals = pd.DataFrame({
    "Horizon": steps,
    "Forecast": smoothing_fit.forecast(horizon).to_numpy(),
    "95% lower": smoothing_bounds[0],
    "95% upper": smoothing_bounds[1],
})
display(smoothing_intervals.head().round(2))

Leaving out trend and seasonality gives simple exponential smoothing. The fit estimates the smoothing weight and initial level.

`anchor="end"` starts each path immediately after the observed history. `nsimulations=horizon` sets the number of future steps, and `repetitions=20_000` sets the number of paths. `error="add"` makes each simulated observation equal its prediction plus a sampled error. `random_errors="bootstrap"` draws those errors from the fitted residuals.

The returned table has twenty rows, one for each horizon, and 20,000 columns, one for each path. This is the transpose of our earlier manual path array. `axis=1` takes percentiles across the paths within each horizon. The 2.5th and 97.5th percentiles become the lower and upper 95% bounds.

The displayed `Forecast` column comes from `.forecast()`. `.to_numpy()` supplies its values in horizon order so the new table can use its own row index. The first displayed row is one step ahead.

For these prices, the fitted smoothing weight is close to one, so the point forecast is close to the latest price. The library resamples its fitted residuals with their empirical mean; our manual random-walk example explicitly centered its error pool. These choices explain why the numerical bounds differ. The underlying procedure remains familiar: resample errors, propagate them through the model, and take percentiles at each horizon.

The same `.simulate()` call also works with the trend, damping, and seasonal specifications we used in the smoothing lecture. See the [Statsmodels simulation documentation](https://www.statsmodels.org/v0.14.6/generated/statsmodels.tsa.holtwinters.HoltWintersResults.simulate.html).

## Check the Intervals Against Later Outcomes

Tuesday's rolling-origin evaluation also works for intervals. Issue each interval using the history available at its origin, then check whether the subsequent observation falls between its bounds.

For a brief example, begin after the first 190 trading observations and issue one-step normal-theory intervals for the remaining 62. At every origin, StatsForecast recalculates the scale from the available changes. This reproduces forecasts we could have issued as the observations arrived.

In [ ]:
checks = forecaster.cross_validation(
    df=data, h=1, n_windows=62, step_size=1, refit=True, level=[80, 95],
)

With `h=1`, each row is one next-observation forecast. `cutoff` records its origin, `y` is the later observed price, and the interval columns contain the bounds issued at that origin. The first forecast uses observations 1–190 to predict observation 191; the last uses 1–251 to predict observation 252.

Calculate coverage as the fraction of outcomes inside the bounds. Also retain average width, since a range must be narrow enough to help with the decision.

In [ ]:
coverage_rows = []
for level in [80, 95]:
    lower = checks[f"Naive-lo-{level}"]
    upper = checks[f"Naive-hi-{level}"]
    inside = checks["y"].between(lower, upper)
    coverage_rows.append({
        "Nominal coverage": f"{level}%",
        "Inside": int(inside.sum()),
        "Forecasts": len(inside),
        "Observed coverage (%)": 100 * inside.mean(),
        "Average width (USD)": (upper - lower).mean(),
    })
coverage = pd.DataFrame(coverage_rows)
display(coverage.round(2))

The 80% intervals contain 55 of 62 outcomes, about 88.7%. The 95% intervals contain 60, about 96.8%. The wider intervals cover more outcomes. Over this short period, both observed coverage rates exceed their nominal levels; with only 62 outcomes, the percentages can fluctuate substantially.

A 95% interval allows some misses even when the model is appropriate. A run of misses on one side, or persistently poor coverage, gives us a reason to examine the forecast and its assumptions. Continue to check by horizon when the task involves several horizons, as it did in the tourism case.

## Using Prediction Intervals

We now have two ways to construct a range around a forecast:

| Construction | How It Represents Future Shocks | How It Obtains Bounds |
| --- | --- | --- |
| Normal theory | A normal distribution with an estimated error scale | Normal quantiles and the horizon-specific forecast-error standard deviation |
| Residual bootstrap | Repeated draws from a representative pool of historical errors | Percentiles of simulated future observations at each horizon |

Both approaches rely on the model and historical evidence remaining useful. A change in the process can make either set of intervals misleading. For the random walk, independent shocks accumulate and uncertainty grows with the square root of the horizon. For seasonal-naive, uncertainty grows as we cross additional seasonal cycles; the first four quarterly horizons can have equal interval widths. ETS and ARIMA propagate shocks according to their own model structure.

When using a forecast, state its target and horizon, show its prediction interval, and check its subsequent performance. A future observation outside the interval is a reason to investigate: it may reflect an unusual event, a data problem, or a change in the process. Repeated misses can reveal a forecast that needs updating.

## Related Problems and Methods

A time column can appear in several kinds of analytical problems. Start by identifying the observational unit, what is recorded over time, and the question you want to answer. Those choices help determine the appropriate method.

### Longitudinal Data and Mixed-Effects Models

**How does a patient’s response change over repeated visits?**

Longitudinal data follow the same people or units over time. Measurements from one person share characteristics and may remain correlated. People can also differ in their starting levels and rates of change.

[Mixed-effects models](https://www.statsmodels.org/stable/mixed_linear.html) combine population-level relationships with variation between people or groups, such as different intercepts or slopes. They help study individual trajectories and associations with predictors while accounting for the repeated-measurement structure.

### Panel Data Models

**Within a store, how do changes in staffing relate to changes in sales?**

Panel data also follow multiple units over time, often on a shared calendar. A high-sales store may always employ more people; a staffing increase within that same store asks a different question.

[Panel models](https://bashtage.github.io/linearmodels/panel/introduction.html) can account for persistent differences between stores and shared time effects, such as a holiday affecting all stores. Longitudinal and panel approaches overlap; the question and assumptions guide the model choice.

### Survival Analysis and Hazard Models

**How long until a machine fails or a customer leaves?**

Here the outcome is the time until an event. A machine still operating when observation ends supplies partial information: its failure time exceeds the period for which we observed it. This is called right censoring.

[Survival analysis](https://mc-stan.org/docs/stan-users-guide/survival.html) incorporates that information. Survival curves describe the probability of remaining event-free beyond a given duration. Hazard models, such as Cox regression, relate predictors to the event rate among units that have remained event-free. The time scale may be time since installation or enrollment.

### Intervention Effects and Causal Inference

**Did a new policy change demand?**

The target is the difference between what happened and what would have happened without the policy. Other changes, existing trends, and seasonality can also affect the outcome.

Interrupted time-series analysis, difference-in-differences, and synthetic-control methods offer ways to investigate intervention effects. They require a defensible comparison or assumptions about the outcome without intervention. [Counterfactual forecasting](https://arxiv.org/abs/1506.00356) can contribute to that comparison when suitable unaffected series are available.

For a new problem, ask: am I predicting a future value, studying repeated measurements, estimating time to an event, or assessing an intervention? More than one may apply. Naming the target and observational unit is the first modeling decision.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).